# RQ2 Code Generation Analysis

**Purpose**: Analyze code generation results from RQ2 experiments comparing Dual-Agent vs Multi-Agent approaches.

**Datasets**: 16 code generation experiments (8 pods × 2 experiments per pod)
- **Dual-Agent (DA)**: 2-agent approach
- **Multi-Agent (MA)**: 4-agent approach  

**Research Questions**:
- **RQ2.1**: How do dual-agent and multi-agent approaches compare in code generation quality (Pass@1)?
- **RQ2.2**: What is the impact of prompting strategy (zero-shot vs few-shot)?
- **RQ2.3**: How do model size (4B vs 30B) and reasoning capabilities (Instruct vs Thinking) affect performance?
- **RQ2.4**: What are the energy efficiency tradeoffs?

**Metrics**: Pass@1 (percentage of samples that pass all test cases)

**Date**: November 17, 2025

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Paths
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'analysis' / 'rq2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"📊 Results Directory: {RESULTS_DIR}")
print(f"💾 Output Directory: {OUTPUT_DIR}")

## 2. Load Code Generation Results

In [ ]:
# Define pod configurations
pod_configs = {
    'pod1': {'model_size': '4B', 'model_type': 'Instruct', 'prompting': 'Zero-shot'},
    'pod2': {'model_size': '4B', 'model_type': 'Thinking', 'prompting': 'Zero-shot'},
    'pod3': {'model_size': '4B', 'model_type': 'Instruct', 'prompting': 'Few-shot'},
    'pod4': {'model_size': '4B', 'model_type': 'Thinking', 'prompting': 'Few-shot'},
    'pod5': {'model_size': '30B', 'model_type': 'Instruct', 'prompting': 'Zero-shot'},
    'pod6': {'model_size': '30B', 'model_type': 'Thinking', 'prompting': 'Zero-shot'},
    'pod7': {'model_size': '30B', 'model_type': 'Instruct', 'prompting': 'Few-shot'},
    'pod8': {'model_size': '30B', 'model_type': 'Thinking', 'prompting': 'Few-shot'},
}

# Load all code generation results
all_results = []

for pod_name, config in pod_configs.items():
    pod_dir = RESULTS_DIR / f'runpod_rq2_{pod_name}'
    
    # Handle nested results directories
    if (pod_dir / 'results').exists():
        pod_dir = pod_dir / 'results'
    
    # Find DA and MA code generation evaluation files
    da_files = list(pod_dir.glob('DA-code-*_evaluation.json'))
    ma_files = list(pod_dir.glob('MA-code-*_evaluation.json'))
    
    for file in da_files + ma_files:
        agent_type = 'Dual-Agent' if 'DA-' in file.name else 'Multi-Agent'
        
        # Load evaluation JSON
        with open(file, 'r') as f:
            eval_data = json.load(f)
        
        # Extract Pass@1 score
        pass_at_1 = eval_data.get('pass@1', eval_data.get('pass_at_1', 0))
        total_samples = eval_data.get('total_samples', eval_data.get('n', 0))
        passed_samples = eval_data.get('passed', eval_data.get('c', 0))
        
        result = {
            'pod': pod_name,
            'agent_type': agent_type,
            'model_size': config['model_size'],
            'model_type': config['model_type'],
            'prompting': config['prompting'],
            'pass_at_1': pass_at_1,
            'pass_at_1_pct': pass_at_1 * 100,
            'total_samples': total_samples,
            'passed_samples': passed_samples,
            'file': file.name
        }
        all_results.append(result)

# Create DataFrame
df_code = pd.DataFrame(all_results)

print(f"\n📊 Loaded {len(df_code)} code generation experiments")
print(f"\nColumns: {list(df_code.columns)}")
df_code.head(10)

## 3. Load Energy Data

In [ ]:
# Load energy data from emissions.csv files
energy_data = []

for pod_name, config in pod_configs.items():
    pod_dir = RESULTS_DIR / f'runpod_rq2_{pod_name}'
    
    emissions_file = pod_dir / 'emissions.csv'
    if not emissions_file.exists():
        emissions_file = pod_dir / 'results' / 'emissions.csv'
    
    if emissions_file.exists():
        df_emissions = pd.read_csv(emissions_file)
        
        total_energy = df_emissions['energy_consumed'].sum()
        total_emissions = df_emissions['emissions'].sum()
        
        energy_data.append({
            'pod': pod_name,
            'total_energy_kwh': total_energy,
            'total_emissions_kg': total_emissions
        })

df_energy = pd.DataFrame(energy_data)
df_code = df_code.merge(df_energy, on='pod', how='left')

# Calculate energy per sample
df_code['energy_per_sample'] = df_code['total_energy_kwh'] / df_code['total_samples']

print("\n✅ Energy data loaded and merged!")
df_code.head()

## 4. RQ2.1: Dual-Agent vs Multi-Agent Comparison

In [ ]:
# Compare DA vs MA performance
agent_comparison = df_code.groupby('agent_type')['pass_at_1_pct'].agg(['mean', 'std', 'min', 'max'])

print("📊 Dual-Agent vs Multi-Agent Pass@1 Performance:\n")
print(agent_comparison.round(2))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
agent_comparison['mean'].plot(kind='bar', ax=axes[0], rot=0, color=['#3498DB', '#E74C3C'], 
                               yerr=agent_comparison['std'], capsize=5)
axes[0].set_title('Code Generation Pass@1: Dual-Agent vs Multi-Agent', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Agent Type', fontsize=12)
axes[0].set_ylabel('Pass@1 (%)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Box plot
df_code.boxplot(column='pass_at_1_pct', by='agent_type', ax=axes[1])
axes[1].set_title('Pass@1 Distribution by Agent Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Agent Type', fontsize=12)
axes[1].set_ylabel('Pass@1 (%)', fontsize=12)
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_code_da_vs_ma_performance.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical test
from scipy import stats
da_pass = df_code[df_code['agent_type'] == 'Dual-Agent']['pass_at_1_pct']
ma_pass = df_code[df_code['agent_type'] == 'Multi-Agent']['pass_at_1_pct']
t_stat, p_value = stats.ttest_ind(da_pass, ma_pass)
print(f"\n📊 T-test (DA vs MA Pass@1): t={t_stat:.3f}, p={p_value:.3f}")
if p_value < 0.05:
    print("   ✅ Statistically significant difference!")
else:
    print("   ⚠️  No statistically significant difference")

## 5. RQ2.2: Prompting Strategy Analysis

In [ ]:
# Compare Zero-shot vs Few-shot
prompting_comparison = df_code.groupby('prompting')['pass_at_1_pct'].agg(['mean', 'std', 'min', 'max'])

print("📊 Zero-shot vs Few-shot Pass@1 Performance:\n")
print(prompting_comparison.round(2))

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
prompting_comparison['mean'].plot(kind='bar', ax=ax, rot=0, color=['#2ECC71', '#E67E22'],
                                   yerr=prompting_comparison['std'], capsize=5)
ax.set_title('Code Generation: Zero-shot vs Few-shot', fontsize=14, fontweight='bold')
ax.set_xlabel('Prompting Strategy', fontsize=12)
ax.set_ylabel('Pass@1 (%)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_code_prompting_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. RQ2.3: Model Configuration Analysis

In [ ]:
# Model size comparison
size_comparison = df_code.groupby('model_size')['pass_at_1_pct'].agg(['mean', 'std'])

print("📊 4B vs 30B Model Performance:\n")
print(size_comparison.round(2))

# Model type comparison
type_comparison = df_code.groupby('model_type')['pass_at_1_pct'].agg(['mean', 'std'])

print("\n📊 Instruct vs Thinking Performance:\n")
print(type_comparison.round(2))

# Combined visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

size_comparison['mean'].plot(kind='bar', ax=axes[0], rot=0, color=['#3498DB', '#9B59B6'],
                              yerr=size_comparison['std'], capsize=5)
axes[0].set_title('Pass@1 by Model Size', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Model Size', fontsize=12)
axes[0].set_ylabel('Pass@1 (%)', fontsize=12)
axes[0].grid(True, alpha=0.3)

type_comparison['mean'].plot(kind='bar', ax=axes[1], rot=0, color=['#E74C3C', '#16A085'],
                              yerr=type_comparison['std'], capsize=5)
axes[1].set_title('Pass@1 by Model Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Model Type', fontsize=12)
axes[1].set_ylabel('Pass@1 (%)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_code_model_configuration.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. RQ2.4: Energy Efficiency Analysis

In [ ]:
# Performance vs Energy Tradeoff
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Plot with different colors for agent types
for agent_type in df_code['agent_type'].unique():
    subset = df_code[df_code['agent_type'] == agent_type]
    marker = 'o' if agent_type == 'Dual-Agent' else '^'
    ax.scatter(
        subset['total_energy_kwh'],
        subset['pass_at_1_pct'],
        s=200,
        alpha=0.7,
        marker=marker,
        label=agent_type
    )

ax.set_xlabel('Total Energy Consumed (kWh)', fontsize=12)
ax.set_ylabel('Pass@1 (%)', fontsize=12)
ax.set_title('Code Generation: Performance vs Energy Tradeoff', fontsize=14, fontweight='bold')
ax.legend(title='Agent Type')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_code_energy_tradeoff.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate efficiency (Pass@1 per kWh)
df_code['efficiency'] = df_code['pass_at_1_pct'] / df_code['total_energy_kwh']
print("\n📊 Top 5 Most Efficient Configurations:")
print(df_code.nlargest(5, 'efficiency')[['pod', 'agent_type', 'pass_at_1_pct', 'total_energy_kwh', 'efficiency']].to_string(index=False))

## 8. Comprehensive Heatmap

In [ ]:
# Create configuration column
df_code['config'] = df_code['model_size'] + '-' + df_code['model_type'] + '-' + df_code['prompting']

# Pivot table for heatmap
heatmap_data = df_code.pivot_table(
    values='pass_at_1_pct',
    index='config',
    columns='agent_type',
    aggfunc='mean'
)

print("📊 Pass@1 Heatmap (Configuration × Agent Type):\n")
print(heatmap_data.round(2))

# Visualization
plt.figure(figsize=(10, 12))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=100, cbar_kws={'label': 'Pass@1 (%)'})
plt.title('RQ2 Code Generation Pass@1: Configuration × Agent Type',
          fontsize=14, fontweight='bold')
plt.xlabel('Agent Type', fontsize=12)
plt.ylabel('Configuration', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_code_configuration_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Summary & Export

In [ ]:
# Export to Excel
excel_file = OUTPUT_DIR / 'rq2_code_generation_analysis.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    df_code.to_excel(writer, sheet_name='All Results', index=False)
    agent_comparison.to_excel(writer, sheet_name='DA vs MA')
    prompting_comparison.to_excel(writer, sheet_name='Prompting Strategy')
    size_comparison.to_excel(writer, sheet_name='Model Size')
    type_comparison.to_excel(writer, sheet_name='Model Type')

print(f"\n✅ Analysis complete!")
print(f"📊 Excel file saved: {excel_file}")
print(f"🖼️  Visualizations saved to: {OUTPUT_DIR}/")

# Display final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\nTotal Experiments Analyzed: {len(df_code)}")
print(f"\nBest Pass@1: {df_code['pass_at_1_pct'].max():.2f}% ({df_code.loc[df_code['pass_at_1_pct'].idxmax(), 'config']})")
print(f"\nAverage Pass@1 by Agent Type:")
print(f"  - Dual-Agent: {df_code[df_code['agent_type'] == 'Dual-Agent']['pass_at_1_pct'].mean():.2f}%")
print(f"  - Multi-Agent: {df_code[df_code['agent_type'] == 'Multi-Agent']['pass_at_1_pct'].mean():.2f}%")
print(f"\nAverage Energy Consumption: {df_code['total_energy_kwh'].mean():.3f} kWh")
print(f"Average CO2 Emissions: {df_code['total_emissions_kg'].mean():.3f} kg")
print("\n" + "="*80)

## 10. Key Findings

**To be filled after analysis:**

### RQ2.1: Dual-Agent vs Multi-Agent
- Pass@1 difference: [TBD]
- Quality of generated code: [TBD]

### RQ2.2: Prompting Strategy
- Zero-shot vs Few-shot impact: [TBD]
- Best practices: [TBD]

### RQ2.3: Model Configuration
- Model size impact: [TBD]
- Reasoning capability impact: [TBD]

### RQ2.4: Energy Efficiency
- Cost-benefit analysis: [TBD]
- Optimal configuration: [TBD]